# Notebook 10: Modern LLM Architectures

## Why This Matters

If you open the LLaMA, Mistral, or Gemma source code and compare it to the original "Attention Is All You Need" transformer, you'll find six significant changes. Each change was motivated by either training stability, inference efficiency, or empirical quality improvement. Understanding these changes — not just what they are, but why they were made — is what separates an engineer who can fine-tune a model from one who can design the next one. This notebook builds a "modern LLM block" incorporating all production improvements and shows how each change affects the parameter count and computational profile of a real 7B model.

## Background

**Original transformer (2017):** Post-LN, GELU/ReLU FFN, sinusoidal PE, full MHA, biases everywhere, weight sharing not always applied.

**Modern LLM block (2023-2024):**
1. **Pre-RMSNorm** instead of Post-LayerNorm
2. **SwiGLU FFN** instead of GELU/ReLU
3. **RoPE** instead of learned or sinusoidal PE
4. **GQA** instead of full MHA
5. **No biases** in linear layers
6. **Weight tying** (LM head = token embedding transpose)

Each change has a clear motivation. Together, they produce a model that trains stably at scale, is parameter-efficient, and runs faster at inference.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
print("Ready.")

## 1. The Six Design Decisions and Their Motivations

| Change | Original (2017) | Modern LLM (2023) | Reason |
|---|---|---|---|
| Normalization | Post-LayerNorm | Pre-RMSNorm | Stable training; faster computation |
| Activation | ReLU/GELU | SwiGLU | Better perplexity across sizes |
| Position | Sinusoidal | RoPE | Relative positions; better extrapolation |
| KV heads | Full MHA (H heads) | GQA (G < H heads) | KV cache memory for long context |
| Biases | Yes in Linear | No biases | Simplicity; slight perf improvement |
| Weight tying | Sometimes | Always | Fewer params; better perplexity |

**Why no biases?** Modern LLMs typically remove bias terms from all linear projections (attention Q,K,V,O and FFN). In practice:
- RMSNorm already handles re-centering; biases in LayerNorm (`beta`) do similar work redundantly
- Removing biases simplifies initialization and reduces parameters slightly
- Empirically: no quality difference, sometimes slightly better

**Why weight tying?** The token embedding $E \in \mathbb{R}^{V \times d}$ and the LM head $W_{\text{out}} \in \mathbb{R}^{d \times V}$ are related: both map between token space and embedding space. Sharing them (setting $W_{\text{out}} = E^T$) reduces parameters by $V \times d$ (for LLaMA-7B: $32000 \times 4096 \approx 131M$ params saved) and often improves perplexity.

In [ ]:
# ── Building blocks ────────────────────────────────────────────────────

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return (x / rms) * self.gamma

class RotaryEmbedding(nn.Module):
    def __init__(self, d_k, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
        self.register_buffer('inv_freq', inv_freq)

    def get_cos_sin(self, seq_len, device):
        t = torch.arange(seq_len, device=device).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        return emb.cos(), emb.sin()

    @staticmethod
    def rotate_half(x):
        half = x.shape[-1] // 2
        return torch.cat([-x[..., half:], x[..., :half]], dim=-1)

    def forward(self, x, offset=0):
        B, H, T, d_k = x.shape
        cos, sin = self.get_cos_sin(offset + T, x.device)
        cos = cos[offset:offset+T].view(1, 1, T, d_k)
        sin = sin[offset:offset+T].view(1, 1, T, d_k)
        return x * cos + self.rotate_half(x) * sin

class SwiGLUFFN(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            d_ff = 64 * math.ceil(int(8/3 * d_model) / 64)
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

print("Building blocks loaded.")

## 2. Modern LLM Attention: GQA + RoPE

In [ ]:
class ModernLLMAttention(nn.Module):
    """
    Modern LLM attention: GQA + RoPE, Pre-RMSNorm, no biases.
    This is the attention pattern used in LLaMA-3, Mistral, Gemma-2.
    """
    def __init__(self, d_model, n_heads, n_kv_heads, max_len=4096):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_k = d_model // n_heads

        # No biases in any linear layer
        self.q_proj = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

        self.rope = RotaryEmbedding(self.d_k)

    def forward(self, x, mask=None, position_offset=0):
        B, T, _ = x.shape

        Q = self.q_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)

        # Apply RoPE to Q and K (not V)
        Q = self.rope(Q, offset=position_offset)
        K = self.rope(K, offset=position_offset)

        # GQA: repeat K,V for each query group
        K = K.repeat_interleave(self.n_rep, dim=1)
        V = V.repeat_interleave(self.n_rep, dim=1)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.o_proj(out)

# Test
B, T, d_model = 2, 32, 256
n_heads, n_kv_heads = 8, 2
attn = ModernLLMAttention(d_model, n_heads, n_kv_heads)
x = torch.randn(B, T, d_model)
out = attn(x)
print(f"Modern attention output: {out.shape}")
attn_params = sum(p.numel() for p in attn.parameters())
print(f"Attention params: {attn_params:,}")

## 3. Full Modern LLM Block

In [ ]:
class ModernLLMBlock(nn.Module):
    """
    Complete modern LLM block incorporating all production improvements:
    Pre-RMSNorm + GQA + RoPE + SwiGLU + no biases
    """
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff=None):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = ModernLLMAttention(d_model, n_heads, n_kv_heads)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = SwiGLUFFN(d_model, d_ff)

    def forward(self, x, mask=None):
        # Pre-RMSNorm + residual for attention
        x = x + self.attn(self.norm1(x), mask=mask)
        # Pre-RMSNorm + residual for FFN
        x = x + self.ffn(self.norm2(x))
        return x

class ModernLLM(nn.Module):
    """
    Minimal modern LLM (LLaMA/Mistral architecture).
    """
    def __init__(self, vocab_size, d_model, n_heads, n_kv_heads, n_layers,
                 max_len=4096, d_ff=None):
        super().__init__()
        self.d_model = d_model
        self.embed_tokens = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            ModernLLMBlock(d_model, n_heads, n_kv_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying
        self.lm_head.weight = self.embed_tokens.weight

        self._init_weights()

    def _init_weights(self):
        std = self.d_model ** -0.5
        nn.init.normal_(self.embed_tokens.weight, mean=0, std=std)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0, std=std)

    def make_causal_mask(self, T, device):
        return torch.triu(torch.ones(T, T, device=device),
                          diagonal=1).bool().unsqueeze(0).unsqueeze(0)

    def forward(self, input_ids):
        B, T = input_ids.shape
        x = self.embed_tokens(input_ids)
        mask = self.make_causal_mask(T, input_ids.device)
        for layer in self.layers:
            x = layer(x, mask=mask)
        x = self.norm(x)
        return self.lm_head(x)

# Build tiny modern LLM
tiny_llm = ModernLLM(
    vocab_size=32000, d_model=256, n_heads=8, n_kv_heads=2,
    n_layers=4, max_len=512
)
tokens = torch.randint(0, 32000, (2, 64))
logits = tiny_llm(tokens)
print(f"Modern LLM output: {logits.shape}")
print(f"Total parameters: {sum(p.numel() for p in tiny_llm.parameters()):,}")
print(f"Weight tied (lm_head.weight is embed_tokens.weight): "
      f"{tiny_llm.lm_head.weight.data_ptr() == tiny_llm.embed_tokens.weight.data_ptr()}")

## 4. Parameter Counting for a Real 7B Model

Let's compute the exact parameter breakdown for a LLaMA-3-style 7B model.

In [ ]:
def llama_param_count(d_model, n_heads, n_kv_heads, n_layers, vocab_size,
                       d_ff=None):
    """Compute parameter breakdown for a LLaMA-style model."""
    d_k = d_model // n_heads
    if d_ff is None:
        d_ff = 64 * math.ceil(int(8/3 * d_model) / 64)

    # Embeddings (shared with LM head via weight tying)
    embed_params = vocab_size * d_model

    # Per transformer layer
    # Attention: Q, K, V, O projections (no bias)
    q_params = d_model * n_heads * d_k
    k_params = d_model * n_kv_heads * d_k
    v_params = d_model * n_kv_heads * d_k
    o_params = d_model * d_model
    attn_params = q_params + k_params + v_params + o_params

    # SwiGLU FFN: gate, up, down (no bias)
    ffn_params = d_model * d_ff * 3  # gate_proj + up_proj + down_proj

    # RMSNorm: just gamma (d_model params each, 2 per layer)
    norm_params = 2 * d_model

    per_layer = attn_params + ffn_params + norm_params
    all_layers = n_layers * per_layer

    # Final norm
    final_norm = d_model

    total = embed_params + all_layers + final_norm
    # Note: lm_head shares with embed (weight tying), so no extra params

    return {
        "embedding": embed_params,
        "per_layer_attn": attn_params,
        "per_layer_ffn": ffn_params,
        "per_layer": per_layer,
        "all_layers": all_layers,
        "total": total,
        "ffn_fraction": n_layers * ffn_params / all_layers,
        "attn_fraction": n_layers * attn_params / all_layers,
        "d_ff": d_ff,
        "kv_cache_per_token_per_layer_bytes": 2 * n_kv_heads * d_k * 2  # FP16
    }

# LLaMA-3-8B-style configuration
config_8b = dict(d_model=4096, n_heads=32, n_kv_heads=8,
                  n_layers=32, vocab_size=128256)
stats = llama_param_count(**config_8b)

print("LLaMA-3-8B Architecture Parameter Breakdown")
print("="*50)
print(f"d_model={config_8b['d_model']}, n_heads={config_8b['n_heads']}, "
      f"n_kv_heads={config_8b['n_kv_heads']}, n_layers={config_8b['n_layers']}")
print(f"d_ff = {stats['d_ff']}")
print(f"\nEmbedding:       {stats['embedding']/1e9:.3f}B ({stats['embedding']/stats['total']*100:.1f}%)")
print(f"All layers:      {stats['all_layers']/1e9:.3f}B ({stats['all_layers']/stats['total']*100:.1f}%)")
print(f"  - Attention:   {config_8b['n_layers']*stats['per_layer_attn']/1e9:.3f}B ({stats['attn_fraction']*100:.1f}%)")
print(f"  - FFN:         {config_8b['n_layers']*stats['per_layer_ffn']/1e9:.3f}B ({stats['ffn_fraction']*100:.1f}%)")
print(f"Total (weight tied): {stats['total']/1e9:.2f}B")
print(f"\nKV cache: {stats['kv_cache_per_token_per_layer_bytes']} bytes/token/layer (FP16)")
print(f"KV cache for 8K context, {config_8b['n_layers']} layers: "
      f"{stats['kv_cache_per_token_per_layer_bytes'] * 8192 * config_8b['n_layers'] / 1e9:.3f} GB")

## 5. Context Length Scaling: Why RoPE + GQA Enables Long Context

The two binding constraints for context length:

**1. KV cache memory (GQA's contribution):**
$$\text{KV cache} = 2 \times L \times H_{\text{kv}} \times T \times d_k \times \text{bytes}$$

For LLaMA-3-8B with 8K context, GQA-8 (4 KV heads):
- KV cache: $2 \times 32 \times 8 \times 8192 \times 128 \times 2 \approx 1.1$GB

With full MHA (32 KV heads): $\approx 4.3$GB — nearly 4× more. At batch size 4: 17GB just for KV cache!

**2. Positional encoding extrapolation (RoPE's contribution):**
RoPE enables context extension via "RoPE scaling" — by adjusting the base frequency:

$$\theta_i^{\text{new}} = \frac{\theta_i^{\text{original}}}{s^{2i/d_k}}$$

where $s$ is the scaling factor. LLaMA-3 uses a more sophisticated variant (YaRN) to extend from 8K to 128K tokens.

In [ ]:
# Visualize KV cache growth with context length for different architectures
seq_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
n_layers, d_k, dtype_bytes = 32, 128, 2

configs_for_plot = [
    ("LLaMA-3-8B (GQA-8,  8 KV heads)",  8),
    ("LLaMA-2-7B (GQA-none, 32 KV heads)", 32),
    ("MQA (1 KV head)",  1),
]

fig, ax = plt.subplots(figsize=(10, 6))
for label, n_kv in configs_for_plot:
    kv_cache_gb = [2 * n_layers * n_kv * T * d_k * dtype_bytes / 1e9 for T in seq_lengths]
    ax.plot(seq_lengths, kv_cache_gb, marker='o', markersize=6, label=label)

ax.axhline(y=16, color='gray', linestyle=':', alpha=0.7, label="16GB GPU memory")
ax.axhline(y=80, color='black', linestyle=':', alpha=0.7, label="80GB GPU memory")
ax.set_xlabel("Sequence length (tokens)")
ax.set_ylabel("KV cache memory (GB)")
ax.set_title("KV Cache Memory vs Context Length (32 layers, d_k=128, FP16)")
ax.legend()
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.set_xticks(seq_lengths)
ax.set_xticklabels([str(T) for T in seq_lengths], rotation=30, ha='right')
plt.tight_layout()
plt.savefig("kv_cache_scaling.png", dpi=120, bbox_inches='tight')
plt.show()

## Summary

### Key Concepts

| Change from Original | Why It Was Made | Who Uses It |
|---|---|---|
| Pre-RMSNorm | Training stability + 20% faster norm | LLaMA, Mistral, Gemma |
| SwiGLU | Better perplexity; gated expressiveness | LLaMA, Mistral, Gemma, Qwen |
| RoPE | Relative positions; context extension | LLaMA, Mistral, Gemma, Falcon |
| GQA (fewer KV heads) | KV cache reduction for long context | LLaMA-2 70B, LLaMA-3, Mistral |
| No biases | Simplicity; slight perf improvement | LLaMA, Mistral, Gemma |
| Weight tying | Fewer params; better perplexity | All modern LLMs |
| FFN fraction | ~67% of transformer layer params | Scales with d_ff |
| KV cache formula | $2 L H_{\text{kv}} T d_k \text{bytes}$ | Critical for inference planning |